# YĀTRĀ AI — Notebook 04: Feature Engineering & Continuous Travel DNA
## 7-D Continuous Personalization & Choice-Set Quality Scoring

This notebook demonstrates:
1. **7-Dimensional Continuous Travel DNA** vector formulation
2. Persona-based Beta distribution sampling
3. Pre-choice relative candidate itinerary scoring:
   - `cost_score`, `time_score`, `carbon_score`, `reliability_score`, `comfort_score`, `transfer_score`, `departure_fit`
4. Empirical correlation analysis verifying preference interactions


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = os.path.dirname(os.getcwd())
if BASE_DIR not in sys.path:
    sys.path.append(BASE_DIR)

from src.feature_engineering import TRAVEL_DNA_FEATURES, NUMERIC_FEATURES, CATEGORICAL_CORE, compute_candidate_scores
print(f"Authoritative Travel DNA Dimensions (7-D): {TRAVEL_DNA_FEATURES}")


### 1. Inspecting the 5,000 Traveller Population & Travel DNA Distributions


In [ ]:
travellers_path = os.path.join(BASE_DIR, "data", "synthetic", "traveller_population.parquet")
df_travellers = pd.read_parquet(travellers_path)
print(f"Traveller Population: {len(df_travellers):,} profiles")
print("Persona Distribution:\n", df_travellers['persona_type'].value_counts())
df_travellers.head()


In [ ]:
# Summary statistics of the 7 continuous Travel DNA coordinates
df_travellers[TRAVEL_DNA_FEATURES].describe().round(4)


### 2. Travel DNA Correlation Heatmap (7x7 Matrix)
Examining empirical correlations between the 7 Travel DNA dimensions across the population.


In [ ]:
plt.figure(figsize=(8, 6))
corr_matrix = df_travellers[TRAVEL_DNA_FEATURES].corr()
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("Travel DNA 7-D Correlation Heatmap", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


### 3. Demonstrating Candidate Quality Scoring
We show how raw itinerary attributes are transformed into relative choice-set scores.


In [ ]:
import yaml
with open(os.path.join(BASE_DIR, "config", "synthetic_generation.yaml"), "r") as f:
    config = yaml.safe_load(f)

# Sample candidate set
sample_session = {"origin_city": "Delhi", "destination_city": "Mumbai", "departure_window": "Morning"}
sample_dna = {"departure_time_flexibility": 0.30}
sample_cands = [
    {
        "mode": "Flight", "raw_cost": 5200.0, "raw_duration": 2.2, "raw_distance": 1150.0,
        "transfer_count": 0, "p_severe_delay": 0.03, "p_cancelled": 0.01, "p_slight_delay": 0.08,
        "cabin_class_or_tier": "Economy", "departure_window": "Morning"
    },
    {
        "mode": "Rail", "raw_cost": 1850.0, "raw_duration": 15.5, "raw_distance": 1380.0,
        "transfer_count": 0, "p_severe_delay": 0.08, "p_cancelled": 0.02, "p_slight_delay": 0.18,
        "cabin_class_or_tier": "3-Tier AC", "departure_window": "Evening"
    }
]

scored = compute_candidate_scores(sample_cands, sample_session, sample_dna, config)
pd.DataFrame(scored)[['mode', 'raw_cost', 'cost_score', 'raw_duration', 'time_score', 'reliability_score', 'comfort_score', 'carbon_score', 'departure_fit']]
